# 浩劫：Colab 自对弈与持续训练

先选择 **运行时 → 更改运行时类型 → GPU**，按顺序运行单元格。源码、Python、Rust 引擎和活跃训练根都放在 `/content`；Drive 只存完整恢复归档。

恢复本机训练：停止本机 GUI，复制整个训练根到 Drive，并在 `RESTORE_SOURCE` 填目录。已有本 Notebook 备份时，留空即可自动选择最近完整备份。单独权重使用 `IMPORT_WEIGHTS`，它建立新优化器；原始对局 JSONL 不是完整恢复点。

说明：[training/COLAB.md](https://github.com/Loremmlel/haojie2/blob/feat/server-colab-training/training/COLAB.md)。选择 GPU 不改写模型、规则或已有优化器；不兼容的恢复点会报出具体原因。

In [ ]:
# @title 1. 配置与 Drive 授权
from pathlib import Path
import asyncio, html, json, os, subprocess, sys, time, urllib.request, urllib.error
from urllib.parse import urlparse

REPO_REF = "feat/server-colab-training" # @param {type:"string"}
DEVICE = "cuda" # @param ["cuda", "cpu"]
TORCH_BACKEND = "auto" # @param ["auto", "cu126", "cu128", "cu130", "cpu"]
LOCAL_ROOT = "/content/haojie-training" # @param {type:"string"}
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/haojie/backups" # @param {type:"string"}
RESTORE_SOURCE = "" # @param {type:"string"}
IMPORT_WEIGHTS = "" # @param {type:"string"}
BACKUP_MINUTES = 10 # @param {type:"integer"}
BACKUP_KEEP = 3 # @param {type:"integer"}
# 0 表示：新实验按硬件生成；恢复实验沿用原值。修改会影响采样/更新节奏。
ENVIRONMENTS = 0 # @param {type:"integer"}
GAMES_PER_ROUND = 0 # @param {type:"integer"}
INFERENCE_BATCH_SIZE = 0 # @param {type:"integer"}
TRAINING_BATCH_SIZE = 0 # @param {type:"integer"}
POOL_MIB = 0 # @param {type:"integer"}
BATCH_WAIT_MS = -1.0 # @param {type:"number"}
SAMPLING_PRECISION = "沿用" # @param ["沿用", "fp32", "bf16"]
TORCH_THREADS = 1 # @param {type:"integer"}

REPO = Path("/content/haojie2")
PYTHON = Path("/content/haojie-venv/bin/python")
ENGINE = Path("/content/haojie-native/release/haojie-engine")
CONNECTION = Path("/content/haojie-connection.json")
LOG = Path("/content/haojie-cloud.log")
assert not (RESTORE_SOURCE and IMPORT_WEIGHTS), "完整恢复和导入权重请选择一种"
assert BACKUP_MINUTES >= 1 and BACKUP_KEEP >= 2 and TORCH_THREADS >= 1
assert Path("/content/drive") not in Path(LOCAL_ROOT).resolve().parents
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
assert Path("/content/drive/MyDrive").is_dir(), "Drive 尚未成功挂载"

In [ ]:
# @title 2. 获取代码与独立运行环境（首次安装；重复执行会保留训练数据）
def run(args, **kwargs):
    return subprocess.run(list(map(str, args)), check=True, **kwargs)

def live_pid(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except (ProcessLookupError, ValueError, TypeError):
        return False

if CONNECTION.exists():
    old = json.loads(CONNECTION.read_text())
    assert not live_pid(old.get("pid")), "训练服务仍在运行；先执行最后的正常退出单元格"
url = "https://github.com/Loremmlel/haojie2.git"
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, url, REPO])
else:
    assert (REPO / ".git").is_dir(), "源码目录已有其他内容，请选择新的源码目录"
    remote = subprocess.check_output(["git", "-C", REPO, "remote", "get-url", "origin"], text=True).strip()
    assert remote == url, "已有源码目录的远端不同；不会覆盖"
    dirty = subprocess.check_output(["git", "-C", REPO, "status", "--porcelain"], text=True)
    assert not dirty.strip(), "源码存在本地改动；先保留这些改动后再更新"
    run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", REPO_REF])
    run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
run([sys.executable, REPO / "scripts/colab/bootstrap.py", "--backend", TORCH_BACKEND,
     "--device", DEVICE])
print("源码版本：", subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip())

## 启动、监控与续训

启动后先停在“已暂停”，核对实际 CPU 配额、GPU 和恢复更新数，再点击“开始”。并发上限 128；建议从自动配置开始，固定每轮局数分别试 4/8/16/32 环境，对比有效样本与完整周期耗时。`batch_size` 是梯度更新批次，`inference_batch_size` 才是自对弈前向批次。

周期备份独立于下面的监控单元格。中断监控只停止显示，不会杀掉训练服务；离开运行时前请执行最后的“正常退出”。Colab VM 被回收后，从 Drive 最近一次**完成**归档恢复；正在运行的对局和 RAM 样本池重新生成。

In [ ]:
# @title 3. 启动本地训练服务（默认暂停，不开放公网端口）
def read_connection():
    info = json.loads(CONNECTION.read_text())
    parsed = urlparse(info["url"])
    assert parsed.scheme == "http" and parsed.hostname == "127.0.0.1" and parsed.port
    assert not parsed.username and not parsed.password and not parsed.query
    return info

def api(path="/api/status", action=None, values=None):
    base = read_connection()["url"]
    headers = {}
    data = None
    if action is not None:
        with urllib.request.urlopen(base + "/api/session", timeout=10) as response:
            token = json.load(response)["token"]
        headers = {"Origin": base, "X-Session-Token": token, "Content-Type": "application/json"}
        data = json.dumps({"action": action, "values": values}).encode()
        path = "/api/command"
    request = urllib.request.Request(base + path, data=data, headers=headers)
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.load(response)

if CONNECTION.exists() and live_pid(read_connection().get("pid")):
    print("已连接现有训练服务：", api()["state"])
else:
    CONNECTION.unlink(missing_ok=True)
    settings = {key: value for key, value in {
        "environments": ENVIRONMENTS, "games_per_round": GAMES_PER_ROUND,
        "inference_batch_size": INFERENCE_BATCH_SIZE, "batch_size": TRAINING_BATCH_SIZE,
        "pool_mib": POOL_MIB,
    }.items() if value != 0}
    if BATCH_WAIT_MS >= 0:
        settings["batch_wait_ms"] = BATCH_WAIT_MS
    if SAMPLING_PRECISION != "沿用":
        settings["sampling_precision"] = SAMPLING_PRECISION
    settings_path = Path("/content/haojie-settings.json")
    settings_path.write_text(json.dumps(settings))
    command = [str(PYTHON), "-m", "haojie_training.cloud", "--engine", str(ENGINE),
               "--root", LOCAL_ROOT, "--device", DEVICE, "--settings", str(settings_path),
               "--backup-dir", DRIVE_BACKUP_DIR, "--backup-seconds", str(BACKUP_MINUTES * 60),
               "--backup-keep", str(BACKUP_KEEP), "--torch-threads", str(TORCH_THREADS),
               "--scratch", "/content/haojie-archive-work", "--connection-file", str(CONNECTION),
               "--paused"]
    if RESTORE_SOURCE:
        command += ["--restore", RESTORE_SOURCE]
    if IMPORT_WEIGHTS:
        command += ["--import-weights", IMPORT_WEIGHTS]
    with LOG.open("a") as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 600
    while not CONNECTION.exists():
        if process.poll() is not None or time.monotonic() > deadline:
            print(LOG.read_text(errors="replace")[-12000:])
            raise RuntimeError("服务未就绪，请按日志处理；训练原件与完成备份保留")
        time.sleep(1)
    print("服务就绪：", api()["state"])
print(json.dumps({key: api().get(key) for key in ("hardware", "restoration", "updates", "config")},
                 ensure_ascii=False, indent=2))

In [ ]:
# @title 4. 控制面板（修改设置前先暂停）
import ipywidgets as widgets
from IPython.display import display, HTML

notice = widgets.Output()
controls = []
def clicked(action):
    def execute(_):
        with notice:
            notice.clear_output(wait=True)
            try:
                print(action, api(action=action))
            except Exception as error:
                print(str(error))
    return execute

for label, action in [("开始", "start"), ("暂停", "pause"), ("保存本地", "save"),
                      ("备份到 Drive", "backup"), ("评测", "evaluate")]:
    button = widgets.Button(description=label)
    button.on_click(clicked(action))
    controls.append(button)
settings_editor = widgets.Textarea(value=json.dumps({}, indent=2),
                                  description="设置覆盖", layout=widgets.Layout(width="90%", height="100px"))
apply = widgets.Button(description="应用设置")
def apply_settings(_):
    with notice:
        notice.clear_output(wait=True)
        try:
            print(api(action="settings", values=json.loads(settings_editor.value)))
        except Exception as error:
            print(str(error))
apply.on_click(apply_settings)
display(widgets.HBox(controls), settings_editor, apply, notice)

In [ ]:
# @title 5. 实时监控（中断此单元格只停止显示）
view = display(HTML("正在读取状态"), display_id=True)
try:
    while True:
        status = api()
        shown = {key: status.get(key) for key in
                 ("state", "error", "device", "updates", "saved_step", "counts", "performance",
                  "resources", "backup", "config")}
        view.update(HTML("<pre style='white-space:pre-wrap'>" +
                         html.escape(json.dumps(shown, ensure_ascii=False, indent=2)) + "</pre>"))
        await asyncio.sleep(5)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("显示已停止，训练与周期备份仍由本地服务执行。")

In [ ]:
# @title 6. 正常退出：暂停 → 保存 → 完成最后一份 Drive 备份
if CONNECTION.exists():
    info = read_connection()
    if live_pid(info["pid"]):
        api(action="shutdown")
        deadline = time.monotonic() + 180
        while live_pid(info["pid"]) and time.monotonic() < deadline:
            if "process" in globals() and process.pid == info["pid"]:
                process.poll()  # 回收自身启动的已退出子进程。
            await asyncio.sleep(1)
        if live_pid(info["pid"]):
            raise RuntimeError("保存或上传仍在进行，请查看日志；这里不会强杀服务")
        if "process" in globals() and process.pid == info["pid"]:
            assert process.wait() == 0, LOG.read_text(errors="replace")[-12000:]
    print(LOG.read_text(errors="replace")[-4000:])
    print("已完成清单：")
    for manifest in sorted(Path(DRIVE_BACKUP_DIR).glob("haojie-*.json")):
        if manifest.name != "haojie-backup-store.json":
            print(manifest.name)